# Notebook 02 — Food-101: learning, representation quality and compute cost

**Click Run All to perform the full experiment.** It benchmarks the two encoders, downloads Food-101 if needed, audits and prepares the data, verifies exact checkpoint recovery, launches paired training in detached tmux, waits while displaying saved progress, and shows the four evaluations and final report.

No mode switches or download toggles are needed. Repeating Run All with the same `RUN_ID` reconnects to an active run or resumes validated saved work. Completed runs display their results.

| Item | Experiment |
|---|---|
| Inputs | Completed Notebook 01 handoff, tagged project definitions, `configs/food101_pilot.yaml`, official Food-101 archive |
| Model | 128×128 RGB, patch 8 → 256 tokens; width 192, depth 4; FMM order 4, leaf 2; FP32 |
| Training | One paired seed; 5,000 successful updates/model; microbatch 4 × accumulation 32 = effective 128; 640,000 image exposures/model |
| Data | 100 training +20 development images/class after audit; 101 classes; official test reserved |
| Evaluation | Initial and trained EMA features for each model; four equal 30-epoch linear probes; training-only standardization |
| Outputs | `artifacts/notebook_02/<RUN_ID>/`, `checkpoints/<RUN_ID>/`, report, plots, protocol and status |
| Runtime | Benchmark, data preparation and preflight first; full training takes hours. Preflight prints a measured estimate before training starts automatically. |
| Kernel | `ahr-go` using `/home/benjamin/workspace/ahr_go/.venv/bin/python` |

Long training runs in detached tmux and continues if this notebook disconnects. The notebook waits for completion and displays progress from saved checkpoints. Interrupting the progress cell stops monitoring, not training.

**Authoring status:** this version has not been executed by the assistant. Run All is active when you choose it.

## 1. What Notebook 01 established

The completed Tiny ImageNet pilot (`20260923T214538Z-9663c15f`) passed its independent artifact review. FMM development top-1 increased from 4.60% to 9.85%; ViT from 5.00% to 8.85%. FMM's +1.00 percentage-point trained-model lead remained inconclusive. Its raw feature effective rank increased 2.53→7.56/192, with substantial remaining concentration. At 64 tokens, its training step was slower.

Those are historical Tiny ImageNet observations, **not Food-101 results**. This notebook starts fresh models; it does not load predecessor weights. The prior source files remain unchanged and their recorded export hashes are checked before any new stage.

[Notebook 01 review](../artifacts/notebook_01/20260923T214538Z-9663c15f/pilot_results_review.md) · [Notebook 02 handoff](../docs/notebook_02_handoff.md)

## 2. Import definitions only

The loader executes only trusted local cells explicitly tagged `ahr-export`. It does not run an upstream notebook's training or generation cells. `CUBLAS_WORKSPACE_CONFIG` is set before importing torch. Restart the kernel after any source change.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot_path = ROOT / "fmm_jepa/bootstrap.ipynb"
_boot_ns = {"__file__": str(_boot_path)}
for _cell in nbformat.read(_boot_path, as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, str(_boot_path), "exec"), _boot_ns)
_boot_ns["install_notebook_package"](ROOT)

# Reload only Food definitions when rerunning in a kernel that opened the earlier notebook.
import sys, importlib
for _name in list(sys.modules):
    if _name.startswith("fmm_jepa.food101_"):
        del sys.modules[_name]
        _package = sys.modules.get("fmm_jepa")
        if _package is not None and hasattr(_package, _name.split(".")[-1]):
            delattr(_package, _name.split(".")[-1])
importlib.invalidate_caches()
from fmm_jepa.food101_workflow import (
    read_config, run_stage, artifact_path, run_all_context, wait_for_food_pilot)

from IPython.display import display, Markdown, Image
import json


## 3. Configuration and automatic resume

The supplied settings run one full paired Food-101 pilot. Keep `RUN_ID` unchanged to resume; choose a new ID only for a new experiment. Run All performs every stage in order and proceeds automatically after the required checks pass.

Configuration and source identities are fixed for each run. Invalid checkpoints or mismatched settings stop with an error; they never silently restart training. The benchmark validates the selected 256-token configuration before preflight. No extra seed or ablation runs automatically.

In [ ]:
RUN_ID = "food101-pilot-001"
CONFIG_FILE = "configs/food101_pilot.yaml"
config = read_config(ROOT, CONFIG_FILE)
artifact = artifact_path(ROOT, RUN_ID)
print("Food-101 full workflow | run:", RUN_ID, flush=True)
print("Artifacts:", artifact)
print("Image size:", config["image_size"], "| effective batch:", config["effective_batch_size"])


In [ ]:
run_context = run_all_context(ROOT, RUN_ID, config)
print("Predecessor:", run_context["predecessor"]["status"])
if run_context["completed"]:
    print("This run is complete; loading its saved results.", flush=True)
elif run_context["session_active"]:
    print("This run is already active; reconnecting to its progress.", flush=True)
elif run_context["skip_preparation"]:
    print("Validated preflight found; resuming the pilot from its checkpoints.", flush=True)
else:
    print("Starting benchmark → data preparation → resume verification → training → evaluation.", flush=True)


## 4. Benchmark larger token grids

This is a **compute diagnostic** on deterministic synthetic RGB images. It cannot establish better visual representations. At each grid, both models see identical images and masks. Measure complete-image encoder forward separately from JEPA forward/backward, AdamW and EMA; report batch size, timing boundaries, peak allocated memory, precision and actual attention backend.

Use an optimized ViT baseline. Record the actual SDPA backend instead of assuming FlashAttention was used. OOM, timeout and errors remain explicit; no unmeasured case is reported as a success. Completed cases are reused only after identity/hash checks. The full-step benchmark uses its stated small batch without production accumulation; the real-data preflight estimates the production budget.

**Benchmark measurements are saved in `compute_benchmark.json`.** Memory may increase sharply at larger grids. The proposed 128×128 setting is not a measured optimum.

In [ ]:
if run_context["skip_preparation"]:
    benchmark = json.loads((artifact / "compute_benchmark.json").read_text())
    print("Using validated saved benchmark.")
else:
    print("Step 1/5: benchmarking 64, 256 and 1,024 image patches…", flush=True)
    benchmark = run_stage(ROOT, RUN_ID, config, "benchmark")
print("Benchmark status:", benchmark["status"])
for row in benchmark["rows"]:
    print({k: row.get(k) for k in ("model_id", "scope", "tokens", "status", "median_seconds", "peak_allocated_bytes")})


## 5. Prepare and audit Food-101

Download the official archive automatically when absent, using its published checksum and a locally recorded SHA256. The adapter records original image dimensions before preprocessing, ordered class mapping, exact selected sample IDs and exact RGB duplicates. Development examples come only from the official training partition. Official test images are reserved; reading them for integrity/overlap auditing does not compute predictions or choose models.

Training uses deterministic sample-ID/occurrence keyed crops and flips. Evaluation resizes the complete RGB image deterministically. Resizing may distort aspect ratio; cropping/upsampling never creates new image information. Original dimension records accompany all resolution claims. No dataset normalization is fitted on development/test data.

The integrity audit writes resumable shards. Interrupted downloads/extraction/audits must satisfy their manifests before reuse. No silent replacement of invalid committed state is permitted.

In [ ]:
if run_context["skip_preparation"]:
    prepared = json.loads((artifact / "split_manifest.json").read_text())
    print("Using prepared Food-101 splits:", prepared["selected_sizes"])
else:
    print("Step 2/5: downloading Food-101 if needed, then auditing and preparing splits…", flush=True)
    prepared = run_stage(ROOT, RUN_ID, config, "prepare", allow_download=True)
    print(json.dumps(prepared, indent=2))


## 6. Register choices and verify exact recovery

The protocol fixes selected data, preprocessing, pairing streams, model parameters, update/probe budgets, source hashes, software/hardware, diagnostics and stopping rules before the main training begins.

The short preflight uses the **actual image size, microbatch, accumulation and full 5,000-update schedule**. For each architecture it compares four uninterrupted updates with two updates followed by state restoration and two further updates. It checks student, predictor, EMA teacher, optimizer, schedule, scaler policy, RNGs, sampler position, data/augmentation/mask traces, counters and non-timing metrics. Failed verification blocks launch.

Checkpoint frequency: every 100 successful updates, epoch boundaries and at most 15 minutes between completed-update saves. Each update is indivisible; recovery replays an interrupted partial update from the preceding valid boundary. Preflight checkpoints are isolated under `checkpoints/<RUN_ID>/preflight/`.

In [ ]:
if run_context["skip_preparation"]:
    preflight = json.loads((artifact / "preflight_certificate.json").read_text())
else:
    print("Step 3/5: verifying exact checkpoint recovery at the actual training configuration…", flush=True)
    preflight = run_stage(ROOT, RUN_ID, config, "preflight")
print("Resume certificate:", preflight["status"])
print("Estimated paired training hours:", preflight.get("estimated_pair_training_hours"))
print("With 25% allowance:", preflight.get("estimated_pair_with_overhead_hours"))
print(preflight.get("estimate_limit", "Inspect the saved certificate."))


## 7. Launch or resume the paired pilot in detached tmux

This cell automatically writes a worker `.ipynb` and starts it using the existing `ahr-go` kernel. The launcher returns immediately and the progress monitor waits for completion; the worker runs independently of the notebook connection. It performs sequential ViT and FMM training, all four probes, raw diagnostics and the report. One writer owns the run. Repeating launch finds the existing session or resumes validated checkpoints; it never silently starts fresh over damaged state.

**Exact resume:** reopen this notebook with the original `RUN_ID` and unchanged config, then click **Run All**. Validated preparation stages are reused, active training is reconnected, and interrupted training resumes from its checkpoints. Do not delete manifests to bypass an error. Successful updates, scheduler/EMA position, latest metric and consumed sampler position are checked on restoration.

Checkpoints: `checkpoints/<RUN_ID>/vit/`, `fmm/`, and `probes/<model_id>/`. Logs/status/protocol: `artifacts/notebook_02/<RUN_ID>/`.

Attach: `tmux attach -t fmm-food101-<RUN_ID>`. Detach: **Ctrl-b, then d**. The actual command/session is written to `launch.json`. Training session names are distinct from the existing AHR jobs.

Storage is the workspace boot disk; instance-deletion policy is unverified. Copy final/best checkpoints off-instance and verify hashes before instance recycling. This notebook does not recycle or destroy anything.

In [ ]:
print("Step 4/5: launching or reconnecting to paired training and evaluation…", flush=True)
launched = run_stage(ROOT, RUN_ID, config, "launch")
print(json.dumps(launched, indent=2))
review = wait_for_food_pilot(ROOT, RUN_ID, config)


## 8. Evaluate representations and inspect quality against cost

The detached worker performs the following fixed analysis:

1. Extract full-image EMA features at initialization and at the final registered update for both models. Pool the final LayerNorm patch outputs; exclude the predictor.
2. Fit standardization on probe-training features only, then train four classifiers with equal budgets and initial weights. Select the best development epoch, earliest tie.
3. Report top-1/top-5 and paired trained-minus-initial and FMM-minus-ViT intervals. One-seed image bootstrap does not measure training-seed or development-selection uncertainty.
4. At initial/middle/final training points, measure raw feature standard deviation, covariance spectrum/effective rank, between-image cosine, patch variation and near/far activity. Advisory rank/variance flags do not certify broadly rich representations.
5. Compare accuracy with measured full-step time and memory, including declared parameter-count differences. Synthetic timing and real-data training timing have separate scopes.

The proposed positive-learning rule requires both complete budgets, four complete probes, a positive lower paired-image interval for FMM's trained-minus-initial top-1, and no registered final degeneration flag. An architectural advantage is a separate question. This run does not automatically authorize or launch more seeds, near-only ablations or later datasets.

In [ ]:
print("Step 5/5: completed results and quality-versus-cost report.")
print("Current status:", review["pilot_status.json"])
comparison = review["comparison_summary.json"]
for name, value in comparison["results"].items():
    print(name, value["metrics"])
print("Learning evidence:", comparison["learning_evidence"])
print("Paired differences:", comparison["deltas"])
report = artifact / "food101_results.md"
display(Markdown(report.read_text()))
for name in ("food101_quality_cost.png", "food101_paired_intervals.png"):
    display(Image(filename=str(artifact / "figures" / name)))


## 9. Interpretation and next decision

**Food-101 results: NOT RUN.** No performance, learning, collapse or readiness outcome is inferred from this notebook's existence or syntax checks. Run All waits for the experiment to finish and then displays its report and plots above.

After the paired pilot, inspect learning gain, raw concentration, matched data traces, checkpoint evidence and measured costs together. Repetition across independent training seeds addresses reproducibility; a near-only comparison addresses the distant branch's contribution. Choose the next experiment from the evidence and its budget.

## References and implementation notes

- [Official Food-101 dataset](https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/)
- [Torchvision Food101 source and published archive checksum](https://docs.pytorch.org/vision/main/_modules/torchvision/datasets/food101.html)
- [PyTorch scaled-dot-product attention backends](https://docs.pytorch.org/docs/main/generated/torch.nn.functional.scaled_dot_product_attention.html)
- [Tagged notebook imports](../docs/notebook_imports.md)
- [Notebook 02 usage and validation limits](../docs/notebook_02_usage.md)

Reusable Food support is in `fmm_jepa/food101_{data,training,evaluation,benchmark,workflow}.ipynb`. Training/evaluation are versioned adaptations of the audited Notebook 01 definitions, with Food-specific data paths, provenance and metadata. Shared FMM geometry/operators/model definitions are unchanged. All authored executable project code remains in notebooks.